# Loka Research Agent - Strands Version

Build a research agent about **Loka** step by step with
**[Strands Agents](https://strandsagents.com/)**.

This notebook is a **guided hands-on exercise** that you will follow throughout the workshop. The agent will be built progressively in three exercises, each adding more capabilities. You will find the following markers in the notebook to guide you:

| Marker               | Meaning                                              |
|----------------------|------------------------------------------------------|
| 🛠️ **Setup**        | Run this once to set up the environment and imports. |
| ✅ **Given**          | Code or instructions provided for you                |
| ✏️ **Your turn**     | Code you need to write to complete the exercise.     |
| 🚀 **Going further** | Optional stretch ideas if you finish early.          |
| 📚 **Hints**         | Links into the Strands docs for more information.    |

The exercises contain detailed instructions on what to do, but you are encouraged to explore and experiment. The goal is to learn by doing, so feel free to modify the code and see how it behaves. Refer to the Strands documentation for more information on the concepts and APIs used in this notebook. Make sure to switch on the `Python` toggle in the top right of documentation web page to see examples that match the code in this notebook.

## 🛠️ Setup

Run this cell once to set up the environment and imports. Make sure you have already configured an `.env` file with your **Anthropic API key** and run `uv sync` to have the project dependencies installed. If you haven't done this yet, follow the instructions in the `README.md` file.

In [1]:
import os, sys
from pathlib import Path

# Find the repo root
ROOT = next(b for b in (Path.cwd(), *Path.cwd().parents) if (b / "shared").is_dir())
sys.path.insert(0, str(ROOT / "shared"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
assert os.getenv("ANTHROPIC_API_KEY"), "Add ANTHROPIC_API_KEY to your .env (copy .env.example)."

from strands import Agent, tool
from strands.models.anthropic import AnthropicModel
from knowledge_base import search_documents, list_topics
from website import search_website

model = AnthropicModel(
    client_args={"api_key": os.environ["ANTHROPIC_API_KEY"]},
    model_id=os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001"),
    max_tokens=1024,
    params={"temperature": 0.3},
)

print("Model ready:", model.config["model_id"])

Model ready: claude-haiku-4-5-20251001


## ✅ The knowledge base (given)

Your agent's knowledge lives in `shared/`, already written for you:
`search_documents(query)`, `list_topics()`, and `search_website(query)`. In the
exercises you'll wrap these as **tools**. Run this to see what they return (no API
key needed):

In [2]:
print(list_topics())
print("\n--- search_documents('learning') ---\n")
print(search_documents("learning"))

The Loka knowledge base covers these topics:
- AWS Innovation Partner of the Year
- Time Off and the 5/4 Friday Schedule
- Fully Remote, Work From Anywhere
- In-Person Connection Despite Being Remote
- Cutting-Edge Client Projects
- Multicultural, Global Team
- Learning and Development
- Culture of Innovation and Internal Initiatives

--- search_documents('learning') ---

## Learning and Development
One of Loka's headline goals for 2026 is to become one of the world's best learning organizations. That means real budget and real time for growth: courses, certifications, and the expectation that you keep leveling up. Learning is treated as part of the job, not a side quest.


In [3]:
print("--- search_website('services') ---\n")
print(search_website("what customers does Loka work with?"))

--- search_website('services') ---

(source: live)

[https://www.loka.com/about]
AWS, Jeff oversees Global Direct Sales at Loka, helping SMB customers unlock value through agentic AI, modernization and cloud migration. A culture of innovation Our team members live by the credo What you develop matters. We work wherever we’re most productive and take every other

[https://www.loka.com/]
17 What Fascinates Sol Rashidi? Navigating AI, balancing the possible with the practical and the art of going rogue. Jump to episode Jump to episode Blog Life at Loka • 7.8.26 Loka's Explorer Program: The Best Decision I Almost Didn't Make What an ML Engineer

[https://www.loka.com/]
Cameo. Their deep GenAI, ML and AWS expertise, paired with industry insight and exceptional customer care, makes them a standout among consultancies." Dom Scandinaro CTO , Cameo “In my 35 years of work experience, I have not come across a consulting partner like Loka. They


## Exercise 1: Basic Agent

An agent is a **model** + **tools** + **instructions**. You hand it a tool and *the model decides* when to call it.

Your job: turn the given `search_documents` function into a tool, build the agent, and run it. The system prompt is written for you.

> 💡 The tool's **docstring** is what the model reads to decide when to use it — write it for the model.

**📚 Hints**
- [Defining tools with `@tool`](https://strandsagents.com/docs/user-guide/concepts/tools/#building--loading-tools)
- [Agent API reference](https://strandsagents.com/docs/api/python/strands.agent.agent/)

### ✏️ Your turn

Build a **basic agent** that can answer questions about Loka's internal knowledge base. The model should decide when to call the tool you create.

In [4]:
from strands import Agent, tool

SYSTEM_PROMPT = """You are the Loka Research Agent, a friendly assistant that \
answers questions about Loka (the company).

- Always answer from the knowledge base via your search tool. Don't rely on prior knowledge about Loka.
- If the knowledge base has no answer, say so instead of guessing.
- Be concise, warm, and a little proud of how great Loka is to work at."""


@tool
def search_knowledge_base(query: str) -> str:
    """Search Loka's internal knowledge base for information about the company,
    its benefits, culture, and how it works.

    Args:
        query: A short natural-language description of what to look for.
    """
    return search_documents(query)


agent = Agent(model=model, tools=[search_knowledge_base], system_prompt=SYSTEM_PROMPT)


response = agent("What is Loka's time-off policy?")


Tool #1: search_knowledge_base
Great question! Loka has a really thoughtful approach to time off:

**The 5/4 Schedule:** Loka operates on a 5/4 schedule, meaning every other Friday is off. That gives you **26 extra days off per year** on top of regular vacation time—roughly one long weekend every two weeks, permanently. 

At Loka, work-life balance isn't just a perk—it's built right into the calendar. Pretty cool, right? 😊

### 🎯 Ask your own

In [6]:
response = agent("What kind of projects does Loka's engineering team typically work on?")


Tool #2: search_knowledge_base
Great question! Loka's engineering team works on some really exciting, cutting-edge projects:

**Cutting-Edge Client Projects:** The team focuses on the newest AI tools and frameworks, including:
- Generative AI
- Agent frameworks
- Modern cloud architectures

You won't be maintaining legacy systems here—you're shipping with the tools that are showing up in this year's tech conference talks. It's genuinely modern, forward-thinking work.

Plus, you get to collaborate with a genuinely multicultural, global team across time zones, which brings fresh perspectives to every project. Pretty exciting stuff! 🚀

### 🚀 Going further

If you finished early or want to explore more, try these ideas:

- **Inspect the run.** The call returns an `AgentResult`. Look at `result.message`,
  `result.metrics` (token usage), and `result.stop_reason`.
- **Use a [built-in Strands tool](https://strandsagents.com/docs/user-guide/concepts/tools/#2-vended-tools)** instead of a custom one. For example, add `strands_tools.calculator` to your agent's `tools=[...]` and ask it something math-related. See the
  [community tools package](https://strandsagents.com/docs/user-guide/concepts/tools/community-tools-package/) for more detail.
- **Change the persona** in `SYSTEM_PROMPT` (formal? pirate?) and re-run.
- **Ask something NOT in the knowledge base** and see if the agent admits it doesn't know.